In [1]:
import pandas as pd

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}".rstrip("0").rstrip(".")
)

# Pipeline 產出物檢視

手動檢視 pipeline 執行後的產出物（parquet、pickle、json），用於除錯和驗證資料品質。

In [2]:
"""設定環境：載入配置與建立 DataCatalog"""
import os
from pathlib import Path
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()))

from recsys_tfb.core.config import ConfigLoader
from recsys_tfb.core.catalog import DataCatalog
from recsys_tfb.core.versioning import resolve_dataset_version, resolve_model_version

config = ConfigLoader("conf", env="local")
data_dir = Path("data")

# 解析版本
dataset_version = resolve_dataset_version(data_dir / "dataset", None)
model_version = resolve_model_version(data_dir / "models", None)

try:
    params_inf = config.get_parameters_by_name("parameters_inference")
    inf_config = params_inf.get("inference", params_inf)
    snap_dates = inf_config.get("snap_dates", [])
    snap_date = snap_dates[0].replace("-", "") if snap_dates else "unknown"
except KeyError:
    snap_date = "unknown"

runtime_params = {
    "dataset_version": dataset_version,
    "model_version": model_version,
    "snap_date": snap_date,
}

# 建立 catalog，model/preprocessor 透過 best symlink 讀取（與 __main__.py 一致）
catalog_config = config.get_catalog_config(runtime_params=runtime_params)
for entry_name in ("model", "preprocessor"):
    if entry_name in catalog_config:
        catalog_config[entry_name]["filepath"] = catalog_config[entry_name]["filepath"].replace(model_version, "best")
catalog = DataCatalog(catalog_config)

print(f"Dataset version: {dataset_version}")
print(f"Model version: {model_version}")
print(f"Snap date: {snap_date}")
print("\n已註冊的資料集：")
for name in catalog.list():
    status = "存在" if catalog.exists(name) else "不存在"
    print(f"  - {name} ({status})")

Dataset version: 2271defb
Model version: 1f43b574
Snap date: 20240331

已註冊的資料集：
  - feature_table (存在)
  - label_table (存在)
  - sample_keys (存在)
  - train_keys (存在)
  - train_dev_keys (存在)
  - val_keys (存在)
  - train_set (存在)
  - train_dev_set (存在)
  - val_set (存在)
  - X_train (存在)
  - y_train (存在)
  - X_train_dev (存在)
  - y_train_dev (存在)
  - X_val (存在)
  - y_val (存在)
  - model (存在)
  - best_params (存在)
  - evaluation_results (存在)
  - preprocessor (存在)
  - category_mappings (存在)
  - scoring_dataset (存在)
  - ranked_predictions (存在)


## Source Data

In [3]:
"""檢視 feature_table（Parquet）"""
if catalog.exists("feature_table"):
    df = catalog.load("feature_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nCount cust_id:\n{df.cust_id.nunique()}")
    print(f"\nCount snap_dates:\n{df.snap_date.unique()}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())
else:
    print("feature_table 不存在，請先執行 dataset building pipeline。")

Shape: (600, 8)

Dtypes:
snap_date            datetime64[ns]
cust_id                      object
total_aum                   float64
fund_aum                    float64
in_amt_sum_l1m              float64
out_amt_sum_l1m             float64
in_amt_ratio_l1m            float64
out_amt_ratio_l1m           float64
dtype: object

Count cust_id:
200

Count snap_dates:
['2024-01-31T00:00:00.000000000' '2024-02-29T00:00:00.000000000'
 '2024-03-31T00:00:00.000000000']

Null 統計:
snap_date            0
cust_id              0
total_aum            0
fund_aum             0
in_amt_sum_l1m       0
out_amt_sum_l1m      0
in_amt_ratio_l1m     0
out_amt_ratio_l1m    0
dtype: int64


,snap_date,cust_id,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
0,2024-01-31,C000001,1202104.3,300971.57,40446.55,168057.34,0.0336,0.1398
1,2024-01-31,C000002,1168094.83,84042.96,13239.03,31761.59,0.0113,0.0272
2,2024-01-31,C000003,1192380.5,8308.68,21470.19,58477.36,0.018,0.049
3,2024-01-31,C000004,139897.14,16064.11,70948.52,35576.96,0.5071,0.2543
4,2024-01-31,C000005,43218.7,2848.59,878.64,29483.6,0.0203,0.6822
5,2024-01-31,C000006,726330.26,246102,88826.99,23012.49,0.1223,0.0317
6,2024-01-31,C000007,704980.35,42944.76,1057.21,975.01,0.0015,0.0014
7,2024-01-31,C000008,1562147.98,395481.14,47877.42,157156.66,0.0306,0.1006
8,2024-01-31,C000009,39647.1,13762.75,2952.64,41669.29,0.0745,1.051
9,2024-01-31,C000010,523280.42,152043.47,16002.5,110398.02,0.0306,0.211


,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
count,600,600,600,600,600,600
mean,518722.8098,125150.5157,50096.9575,42286.2271,0.4404,0.377
std,534247.8017,160611.4792,51677.4001,42658.8192,1.9377,1.6395
min,690.48,83.89,24.21,44.19,0.0001,0.0002
25%,159748.475,22836.9625,14033.1075,11118.6925,0.0318,0.0269
50%,361555.145,70106.705,32513.165,28804.505,0.0961,0.0799
75%,696293.795,164306.8125,69245.105,57599.99,0.3046,0.2197
max,3807332.96,1528172.55,475905.99,295499.89,29.3597,26.3775


In [4]:
"""檢視 label_table（Parquet）"""
if catalog.exists("label_table"):
    df = catalog.load("label_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nCount cust_id:\n{df.cust_id.nunique()}")
    print(f"\nCount snap_date:\n{df.snap_date.unique()}")
    print(f"\nCount prod_name:\n{df.prod_name.nunique()}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())

    # Label 分佈
    if "label" in df.columns:
        print(f"\nLabel 分佈:\n{df['label'].value_counts()}")
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
        # 各產品正樣本比例
        pos_rate = df.groupby("prod_name")["label"].mean().sort_values(ascending=False)
        print(f"\n各產品正樣本比例:\n{pos_rate}")
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
else:
    print("label_table 不存在，請先執行 dataset building pipeline。")

Shape: (3000, 7)

Dtypes:
snap_date           datetime64[ns]
cust_id                     object
cust_segment_typ            object
apply_start_date    datetime64[ns]
apply_end_date      datetime64[ns]
label                        int64
prod_name                   object
dtype: object

Count cust_id:
200

Count snap_date:
['2024-01-31T00:00:00.000000000' '2024-02-29T00:00:00.000000000'
 '2024-03-31T00:00:00.000000000']

Count prod_name:
5

Null 統計:
snap_date           0
cust_id             0
cust_segment_typ    0
apply_start_date    0
apply_end_date      0
label               0
prod_name           0
dtype: int64


,snap_date,cust_id,cust_segment_typ,apply_start_date,apply_end_date,label,prod_name
0,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,fx
1,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,usd
2,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,stock
3,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,bond
4,2024-01-31,C000001,mass,2024-02-01,2024-03-01,1,mix
5,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,fx
6,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,usd
7,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,stock
8,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,bond
9,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,mix


,label
count,3000
mean,0.099
std,0.2987
min,0
25%,0
50%,0
75%,0
max,1



Label 分佈:
0    2703
1     297
Name: label, dtype: int64

Prod_name 分佈:
fx       600
usd      600
stock    600
bond     600
mix      600
Name: prod_name, dtype: int64

各產品正樣本比例:
prod_name
mix     0.1117
usd     0.1083
bond    0.0967
fx      0.0933
stock    0.085
Name: label, dtype: float64

Snap_date 分佈:
2024-01-31    1000
2024-02-29    1000
2024-03-31    1000
Name: snap_date, dtype: int64


## Dataset Pipeline

In [5]:
"""載入 train_set、train_dev_set、val_set 並顯示 shape"""
import pandas as pd

splits = {}
for name in ["train_set", "train_dev_set", "val_set"]:
    if catalog.exists(name):
        splits[name] = catalog.load(name)
        print(f"{name}: {splits[name].shape}")
    else:
        print(f"{name} 不存在，請先執行 dataset pipeline。")

train_set: (1000, 13)
train_dev_set: (1000, 13)
val_set: (1000, 13)


In [6]:
"""檢視 split keys：sample_keys、train_keys、train_dev_keys、val_keys"""
for name in ["sample_keys", "train_keys", "train_dev_keys", "val_keys"]:
    if catalog.exists(name):
        df = catalog.load(name)
        print(f"{name}: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        print(f"  count cust_id: {df.cust_id.nunique()}")
        print(f"  snap_date: {df.snap_date.unique()}")
        display(df.head(5))
    else:
        print(f"{name} 不存在，請先執行 dataset pipeline。")
    print()

sample_keys: (600, 2)
  Columns: ['snap_date', 'cust_id']
  count cust_id: 200
  snap_date: ['2024-01-31T00:00:00.000000000' '2024-02-29T00:00:00.000000000'
 '2024-03-31T00:00:00.000000000']


,snap_date,cust_id
0,2024-01-31,C000096
1,2024-01-31,C000016
2,2024-01-31,C000031
3,2024-01-31,C000159
4,2024-01-31,C000129



train_keys: (200, 2)
  Columns: ['snap_date', 'cust_id']
  count cust_id: 200
  snap_date: ['2024-01-31T00:00:00.000000000']


,snap_date,cust_id
0,2024-01-31,C000096
1,2024-01-31,C000016
2,2024-01-31,C000031
3,2024-01-31,C000159
4,2024-01-31,C000129



train_dev_keys: (200, 2)
  Columns: ['snap_date', 'cust_id']
  count cust_id: 200
  snap_date: ['2024-02-29T00:00:00.000000000']


,snap_date,cust_id
0,2024-02-29,C000177
1,2024-02-29,C000034
2,2024-02-29,C000122
3,2024-02-29,C000181
4,2024-02-29,C000103



val_keys: (200, 2)
  Columns: ['snap_date', 'cust_id']
  count cust_id: 200
  snap_date: ['2024-03-31T00:00:00.000000000']


,snap_date,cust_id
0,2024-03-31,C000001
1,2024-03-31,C000002
2,2024-03-31,C000003
3,2024-03-31,C000004
4,2024-03-31,C000005


In [7]:
"""基本統計比較：數值特徵 describe() — splits vs source feature_table"""
num_cols = ["total_aum", "fund_aum", "in_amt_sum_l1m", "out_amt_sum_l1m",
            "in_amt_ratio_l1m", "out_amt_ratio_l1m"]

source_ft = catalog.load("feature_table")

desc_parts = pd.concat({"source_feature_table": source_ft[num_cols].describe()}, axis=1)
display(desc_parts)

for name, df in splits.items():
    desc_parts = {}
    cols = [c for c in num_cols if c in df.columns]
    if cols:
        desc_parts[name] = df[cols].describe()

    desc_parts = pd.concat(desc_parts, axis=1)
    display(desc_parts)

source_feature_table                                             \
                 total_aum    fund_aum in_amt_sum_l1m out_amt_sum_l1m   
count                  600         600            600             600   
mean           518722.8098 125150.5157     50096.9575      42286.2271   
std            534247.8017 160611.4792     51677.4001      42658.8192   
min                 690.48       83.89          24.21           44.19   
25%             159748.475  22836.9625     14033.1075      11118.6925   
50%             361555.145   70106.705      32513.165       28804.505   
75%             696293.795 164306.8125      69245.105        57599.99   
max             3807332.96  1528172.55      475905.99       295499.89   

                                          
      in_amt_ratio_l1m out_amt_ratio_l1m  
count              600               600  
mean            0.4404             0.377  
std             1.9377            1.6395  
min             0.0001            0.0002  
25%             0.0318            0.0269  
50%             0.0961            0.0799  
75%             0.3046            0.2197  
max            29.3597           26.3775

train_set                                                              \
        total_aum    fund_aum in_amt_sum_l1m out_amt_sum_l1m in_amt_ratio_l1m   
count        1000        1000           1000            1000             1000   
mean  482058.9789  116105.152     52004.1389       41593.088           0.3276   
std   456050.0218  133974.255     52018.8349      45902.5204           0.7453   
min      10569.88      615.63         446.72           44.19           0.0011   
25%   150631.8675  23756.3325     14812.6425       8084.8075           0.0336   
50%     342450.85    67741.66      35176.565       25593.875           0.0961   
75%    672740.495 162236.5325     71532.2775        58167.79           0.3354   
max    2715794.84   815702.44      308454.98       295499.89           8.4022   

                         
      out_amt_ratio_l1m  
count              1000  
mean             0.2374  
std              0.4756  
min              0.0002  
25%              0.0241  
50%              0.0781  
75%               0.226  
max               4.593

train_dev_set                                             \
          total_aum    fund_aum in_amt_sum_l1m out_amt_sum_l1m   
count          1000        1000           1000            1000   
mean    506956.2948 121936.4433     50507.4123      41579.3439   
std     521373.0909 151778.5253     48519.0499      38236.2198   
min         2870.75       83.89          24.21           296.4   
25%     156020.8325  22312.6975       15550.47       14043.875   
50%      350333.105    66581.84      32500.405        29905.09   
75%     710197.4425 163385.0125       70027.95      54156.0025   
max      3735337.42    776460.1      273001.95       185943.87   

                                          
      in_amt_ratio_l1m out_amt_ratio_l1m  
count             1000              1000  
mean            0.5056             0.511  
std             2.3106            2.3312  
min             0.0001            0.0011  
25%             0.0309            0.0307  
50%             0.1146            0.0707  
75%             0.3303            0.2038  
max            24.7893           26.3775

val_set                                                              \
        total_aum    fund_aum in_amt_sum_l1m out_amt_sum_l1m in_amt_ratio_l1m   
count        1000        1000           1000            1000             1000   
mean  567153.1556 137409.9517     47779.3213      43686.2496            0.488   
std     609956.71 190021.0278     54194.7484      43401.4787           2.3114   
min        690.48      144.39         592.62          200.08           0.0012   
25%     178951.04   27823.875     12685.6025       10557.595           0.0313   
50%    376605.865   77823.275      30582.125        30782.29           0.0883   
75%   701308.6375 165029.7275     64875.8225        62308.84           0.2345   
max    3807332.96  1528172.55      475905.99       223432.22          29.3597   

                         
      out_amt_ratio_l1m  
count              1000  
mean             0.3826  
std              1.5362  
min              0.0005  
25%              0.0249  
50%              0.0848  
75%              0.2169  
max             18.9251

In [8]:
"""Label 分佈比較：各 split 正樣本率 vs source label_table"""
source_lt = catalog.load("label_table")

# Overall 正樣本率
overall_rates = {"source_label_table": source_lt["label"].mean()}
for name, df in splits.items():
    if "label" in df.columns:
        overall_rates[name] = df["label"].mean()
print("Overall 正樣本率：")
for k, v in overall_rates.items():
    print(f"  {k}: {v:.4f}")

# Per-product 正樣本率
print("\nPer-product 正樣本率：")
per_prod_parts = {"source_label_table": source_lt.groupby("prod_name")["label"].mean()}
for name, df in splits.items():
    if "label" in df.columns and "prod_name" in df.columns:
        per_prod_parts[name] = df.groupby("prod_name")["label"].mean()

per_prod = pd.DataFrame(per_prod_parts)
display(per_prod)

Overall 正樣本率：
  source_label_table: 0.0990
  train_set: 0.1140
  train_dev_set: 0.0950
  val_set: 0.0880

Per-product 正樣本率：


,source_label_table,train_set,train_dev_set,val_set
prod_name,,,,
bond,0.0967,0.105,0.1,0.085
fx,0.0933,0.105,0.085,0.09
mix,0.1117,0.105,0.125,0.105
stock,0.085,0.115,0.07,0.07
usd,0.1083,0.14,0.095,0.09


In [9]:
"""檢視 preprocessed arrays：X_train/y_train、X_train_dev/y_train_dev、X_val/y_val"""
import numpy as np

for prefix in ["train", "train_dev", "val"]:
    x_name = f"X_{prefix}"
    y_name = f"y_{prefix}"
    if catalog.exists(x_name):
        X = catalog.load(x_name)
        if isinstance(X, np.ndarray):
            print(f"{x_name}: shape={X.shape}, dtype={X.dtype}")
        else:
            print(f"{x_name}: shape={X.shape}, type={type(X).__name__}")
            print(f"  Dtypes:\n{X.dtypes}")
    else:
        print(f"{x_name} 不存在，請先執行 dataset pipeline。")
    if catalog.exists(y_name):
        y = catalog.load(y_name)
        if isinstance(y, np.ndarray):
            print(f"{y_name}: shape={y.shape}, dtype={y.dtype}")
            if y.size > 0:
                print(f"  正樣本率: {y.mean():.4f}")
        else:
            print(f"{y_name}: shape={y.shape}, type={type(y).__name__}")
            if hasattr(y, 'mean'):
                mean_val = y.mean()
                if hasattr(mean_val, 'item'):
                    mean_val = mean_val.item()
                print(f"  正樣本率: {mean_val:.4f}")
    else:
        print(f"{y_name} 不存在，請先執行 dataset pipeline。")
    print()

X_train: shape=(1000, 7), type=DataFrame
  Dtypes:
prod_name               int8
total_aum            float64
fund_aum             float64
in_amt_sum_l1m       float64
out_amt_sum_l1m      float64
in_amt_ratio_l1m     float64
out_amt_ratio_l1m    float64
dtype: object
y_train: shape=(1000,), dtype=int64
  正樣本率: 0.1140

X_train_dev: shape=(1000, 7), type=DataFrame
  Dtypes:
prod_name               int8
total_aum            float64
fund_aum             float64
in_amt_sum_l1m       float64
out_amt_sum_l1m      float64
in_amt_ratio_l1m     float64
out_amt_ratio_l1m    float64
dtype: object
y_train_dev: shape=(1000,), dtype=int64
  正樣本率: 0.0950

X_val: shape=(1000, 7), type=DataFrame
  Dtypes:
prod_name               int8
total_aum            float64
fund_aum             float64
in_amt_sum_l1m       float64
out_amt_sum_l1m      float64
in_amt_ratio_l1m     float64
out_amt_ratio_l1m    float64
dtype: object
y_val: shape=(1000,), dtype=int64
  正樣本率: 0.0880



## Training Pipeline

In [11]:
"""檢視 category_mappings（JSON）"""
if catalog.exists("category_mappings"):
    mappings = catalog.load("category_mappings")
    print(f"Type: {type(mappings)}")
    print(f"\n內容：")
    print(mappings)
else:
    print("category_mappings 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

內容：
{'prod_name': ['bond', 'fx', 'mix', 'stock', 'usd']}


In [12]:
"""檢視 preprocessor（Pickle）"""
if catalog.exists("preprocessor"):
    preprocessor = catalog.load("preprocessor")
    print(f"Type: {type(preprocessor)}")
    if isinstance(preprocessor, dict):
        print(f"\nKeys: {list(preprocessor.keys())}")
        for k, v in preprocessor.items():
            print(f"\n--- {k} ---")
            print(f"Type: {type(v)}")
            print(v)
    else:
        if hasattr(preprocessor, "get_params"):
            print(f"\nParams:\n{preprocessor.get_params()}")

else:
    print("preprocessor 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

Keys: ['feature_columns', 'categorical_columns', 'category_mappings', 'drop_columns']

--- feature_columns ---
Type: <class 'list'>
['prod_name', 'total_aum', 'fund_aum', 'in_amt_sum_l1m', 'out_amt_sum_l1m', 'in_amt_ratio_l1m', 'out_amt_ratio_l1m']

--- categorical_columns ---
Type: <class 'list'>
['prod_name']

--- category_mappings ---
Type: <class 'dict'>
{'prod_name': ['bond', 'fx', 'mix', 'stock', 'usd']}

--- drop_columns ---
Type: <class 'list'>
['snap_date', 'cust_id', 'label', 'apply_start_date', 'apply_end_date', 'cust_segment_typ']


In [13]:
"""檢視 model（Pickle）"""
if catalog.exists("model"):
    model = catalog.load("model")
    print(f"Type: {type(model)}")
    if hasattr(model, "get_params"):
        print(f"\nParams:\n{model.get_params()}")

    # LightGBM feature importance
    if hasattr(model, "feature_importances_") and hasattr(model, "feature_name_"):
        import pandas as pd
        importance = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        print(f"\nTop 20 Feature Importance:")
        display(importance.head(20))
else:
    print("model 不存在，請先執行 training pipeline。")

Type: <class 'lightgbm.basic.Booster'>


In [14]:
"""檢視 best_params（JSON）：Optuna 搜出的最佳超參數"""
if catalog.exists("best_params"):
    import json
    best_params = catalog.load("best_params")
    print(f"Type: {type(best_params)}")
    print(f"\n最佳超參數：")
    print(json.dumps(best_params, indent=2, ensure_ascii=False))
else:
    print("best_params 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

最佳超參數：
{
  "learning_rate": 0.07206796917155728,
  "num_leaves": 50,
  "max_depth": 3,
  "min_child_samples": 89,
  "subsample": 0.87937861683409,
  "colsample_bytree": 0.7776788594864499
}


In [15]:
"""檢視 evaluation_results（JSON）：模型評估結果（mAP、per-product AP 等）"""
if catalog.exists("evaluation_results"):
    import json
    eval_results = catalog.load("evaluation_results")
    print(f"Type: {type(eval_results)}")
    print(f"\n評估結果：")
    print(json.dumps(eval_results, indent=2, ensure_ascii=False))
else:
    print("evaluation_results 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

評估結果：
{
  "overall_map": 0.4827160493827161,
  "per_product_ap": {
    "bond": 0.09413754926691581,
    "fx": 0.08746168668060286,
    "mix": 0.12236111679790153,
    "stock": 0.1443124766697356,
    "usd": 0.09172137345561115
  },
  "n_queries": 200,
  "n_excluded_queries": 128
}


## Inference Pipeline

In [16]:
"""檢視 scoring_dataset（Parquet）：推論用的特徵資料集"""
if catalog.exists("scoring_dataset"):
    df = catalog.load("scoring_dataset")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))

    # snap_date 分佈
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
    # prod_name 分佈
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
else:
    print("scoring_dataset 不存在，請先執行 inference pipeline。")

Shape: (1000, 9)

Dtypes:
snap_date            datetime64[ns]
cust_id                      object
prod_name                    object
total_aum                   float64
fund_aum                    float64
in_amt_sum_l1m              float64
out_amt_sum_l1m             float64
in_amt_ratio_l1m            float64
out_amt_ratio_l1m           float64
dtype: object

Null 統計:
snap_date            0
cust_id              0
prod_name            0
total_aum            0
fund_aum             0
in_amt_sum_l1m       0
out_amt_sum_l1m      0
in_amt_ratio_l1m     0
out_amt_ratio_l1m    0
dtype: int64


,snap_date,cust_id,prod_name,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
0,2024-03-31,C000001,bond,176432.31,88112.82,56583.48,5241.63,0.3207,0.0297
1,2024-03-31,C000001,fx,176432.31,88112.82,56583.48,5241.63,0.3207,0.0297
2,2024-03-31,C000001,mix,176432.31,88112.82,56583.48,5241.63,0.3207,0.0297
3,2024-03-31,C000001,stock,176432.31,88112.82,56583.48,5241.63,0.3207,0.0297
4,2024-03-31,C000001,usd,176432.31,88112.82,56583.48,5241.63,0.3207,0.0297
5,2024-03-31,C000002,bond,14359.38,1607.86,98144.04,5710.31,6.8348,0.3977
6,2024-03-31,C000002,fx,14359.38,1607.86,98144.04,5710.31,6.8348,0.3977
7,2024-03-31,C000002,mix,14359.38,1607.86,98144.04,5710.31,6.8348,0.3977
8,2024-03-31,C000002,stock,14359.38,1607.86,98144.04,5710.31,6.8348,0.3977
9,2024-03-31,C000002,usd,14359.38,1607.86,98144.04,5710.31,6.8348,0.3977



Snap_date 分佈:
2024-03-31    1000
Name: snap_date, dtype: int64

Prod_name 分佈:
bond     200
fx       200
mix      200
stock    200
usd      200
Name: prod_name, dtype: int64


In [17]:
"""檢視 ranked_predictions（Parquet）：排序後的推薦結果"""
if catalog.exists("ranked_predictions"):
    df = catalog.load("ranked_predictions")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))

    # 分數分佈統計
    if "score" in df.columns:
        print(f"\n分數分佈：")
        display(df["score"].describe())

    # 各產品平均分數
    if "prod_name" in df.columns and "score" in df.columns:
        avg_score = df.groupby("prod_name")["score"].mean().sort_values(ascending=False)
        print(f"\n各產品平均分數:\n{avg_score}")

    # 排名分佈：確認每位客戶有完整 1~N 排名
    if "rank" in df.columns and "cust_id" in df.columns:
        ranks_per_cust = df.groupby("cust_id")["rank"].agg(["min", "max", "count"])
        print(f"\n每位客戶排名統計：")
        display(ranks_per_cust.describe())
else:
    print("ranked_predictions 不存在，請先執行 inference pipeline。")

Shape: (1000, 5)

Dtypes:
snap_date    datetime64[ns]
cust_id              object
prod_code            object
score               float64
rank                  int64
dtype: object

Null 統計:
snap_date    0
cust_id      0
prod_code    0
score        0
rank         0
dtype: int64


,snap_date,cust_id,prod_code,score,rank
0,2024-03-31,C000001,bond,0.1222,1
1,2024-03-31,C000001,fx,0.1222,2
2,2024-03-31,C000001,mix,0.1222,3
3,2024-03-31,C000001,stock,0.1222,4
4,2024-03-31,C000001,usd,0.1222,5
5,2024-03-31,C000002,bond,0.1222,1
6,2024-03-31,C000002,fx,0.1222,2
7,2024-03-31,C000002,mix,0.1222,3
8,2024-03-31,C000002,stock,0.1222,4
9,2024-03-31,C000002,usd,0.1222,5



分數分佈：


count     1000
mean    0.1137
std     0.0036
min     0.1076
25%     0.1113
50%     0.1139
75%     0.1154
max     0.1222
Name: score, dtype: float64


每位客戶排名統計：


,min,max,count
count,200,200,200
mean,1,5,5
std,0,0,0
min,1,5,5
25%,1,5,5
50%,1,5,5
75%,1,5,5
max,1,5,5
